# 策略概述

## 一句話說明

現役的 DL-THR 名字裡有「DRL」，但它**不是強化學習**：每期 9 個候選門檻的報酬事後
都能精確算出，模型等於拿到完整答案卷，直接用監督回歸學就好。本策略把這張答案卷
收走——**只讓模型看到自己實際選的那一格結果**，其餘八格不告訴它，並改用
$\varepsilon$-greedy 在「照經驗選」與「隨機試」之間取捨。

除此之外，動作選單、狀態特徵、網路結構、滾動前向切分**逐位元相同**。

> 它回答一個現有文獻沒問過的問題：**反事實標籤值多少錢？**
> 若拿掉答案卷後績效顯著下滑，代表 DL-THR 的增益有相當比例來自
> 「這個問題碰巧是全資訊的」，而非學習演算法本身。


# 輸入值卡

::: {.callout-important}

### 這個模型吃什麼、吐什麼

| 項目 | 內容 |
| :--- | :--- |
| **輸入特徵** | **12 維**，全部由**形成期**視窗算出（交易期開始前即已確定，無前視）。單位不一，各自縮放並截尾至約 $[-3,3]$。與 DL-THR 共用同一個函式，見該本階段 2 |
| **輸出** | 9 個動作的預測報酬（%），決策取其一 |
| **動作選單** | SKIP ＋ 8 組 $(entry\_z, exit\_z) \in \{1.5, 2.0, 2.5, 3.0\} \times \{0.0, 0.5\}$ |
| **網路** | MLP 12 → 64 → 64 → 9（ReLU），Adam，學習率 $10^{-3}$ |
| **訓練標籤** | ⚠ **只有實際選中的那一個動作**的報酬（部分回饋）。DL-THR 是 9 個全給 |
| **損失** | ⚠ **遮罩 MSE**——只有選中動作對應的輸出單元收到梯度，其餘八個不動 |
| **決策** | ⚠ $\varepsilon$-greedy：以機率 $\varepsilon$ 均勻隨機、否則取 $\arg\max$。DL-THR 是恆 $\arg\max$ |
| **探索率 $\varepsilon$** | 掃三組：$0.05$ 常數、$0.10$ 常數、$0.20$ 線性衰減至 $0.02$（2000 次決策走完） |
| **啟用學習門檻** | 可用樣本 $\ge 200$；未達時貪婪臂退回基準動作，但**探索照常進行** |
| **每期訓練** | 40 epoch，批次上限 4096 |
| **隨機性** | ⚠ **兩層**：網路初始化／批次洗牌，再加上探索抽樣。**不固定種子**，以五輪獨立重跑的中位數與全距報告 |

:::

::: {.callout-warning}

### 它比 DL-THR 少了一項結構性保證

DL-THR 保證「訓練樣本不足時退回基準動作 → 暖身期行為 $\equiv$ Z-Score」。

本策略**無法**提供這項保證。若暖身期一律選基準，模型就只會觀測到基準動作的報酬，
其餘八個動作永遠沒有樣本、永遠學不到。**部分回饋強迫 agent 從第一期就得探索**，
暖身期因此必然偏離基準。

這不是實作缺陷，而是部分回饋的固有代價——也正是本對照要量化的東西之一。

:::


# 策略架構

```{mermaid}
flowchart LR
  A["量身特徵<br/>12 個形成期性質"] --> B{"ε-greedy"}
  B -->|"1−ε：照經驗"| C["argmax 預測報酬"]
  B -->|"ε：隨機試"| D["均勻抽 1/9"]
  C --> E["執行<br/>用選定門檻跑 Z-Score 狀態機"]
  D --> E
  E --> F["損益"]
  F -.只回饋這一格.-> G["部分回饋樣本<br/>(特徵, 動作, 報酬)"]
  G -.遮罩 MSE.-> C
```

## 與 DL-THR 的差異表（單一變因）

| 環節 | DL-THR（現役） | RL-THR（本策略） |
| :--- | :--- | :--- |
| 動作選單 | 9 個 | **相同** |
| 狀態特徵 | 12 維形成期 | **相同**（共用同一函式） |
| 網路 | MLP 12→64→64→9 | **相同** |
| walk-forward 切分 | 期 $k$ 只用 $t_e < trade\_start_k$ | **相同** |
| 費用／狀態機／損益會計 | 與 Z-Score 同 | **相同** |
| **訓練標籤** | 9 個動作全部反事實回算 | **只有選中的那一個** |
| **損失** | 全 9 維 MSE | **遮罩 MSE（1 維）** |
| **決策** | 恆 $\arg\max$ | **$\varepsilon$-greedy** |
| **暖身期保證** | $\equiv$ Z-Score | **無**（必須探索） |

實作於 `strategies/trading/rl_threshold_trading.py`；特徵函式直接引用
DL-THR 的 `_pair_features`，避免兩邊各寫一份而在日後悄悄漂走。


# 參考文獻與引用對應


## 文獻 1：Sutton & Barto (2018)

> Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

**參考部分**：

- 第 2 章 **多臂拉霸機（bandit）** 與**情境式拉霸機（contextual bandit）**：
  在「只觀察得到所選動作之報酬」的部分回饋下，$\varepsilon$-greedy 的探索－利用權衡
- MDP 形式化 $(S, A, P, R, \gamma)$ 中，折扣因子 $\gamma$ 之所以必要，
  是因為當前動作會改變後續狀態

**為何參考**：

- 本策略是 contextual bandit 的直接實作：狀態為 12 維形成期特徵，動作為 9 個門檻，
  報酬僅在所選動作上可觀察
- 同時據以說明本策略**為何沒有 $\gamma$**（見階段 4）——這是刻意而非簡化


## 文獻 2：Kim & Kim (2019)

> Kim, T., & Kim, H. Y. (2019). Optimizing the pairs-trading strategy using deep reinforcement learning with trading and stop-loss boundaries. *Complexity*, 2019, Article 3582516.

**參考部分**：

- 門檻選擇式動作空間：學習器輸出交易邊界而非逐日持倉

**為何參考**：

- 動作空間與 DL-THR 完全共用，故本對照的變因純粹落在「回饋型態」上
- 該文獻採用的是真正的強化學習；本策略即是在**同一動作空間**上把
  DL-THR 還原成該文獻的回饋設定，使兩者可直接相比


# 各階段行為

引擎對每個交易期的每組配對依序執行：特徵萃取 → 訓練 → $\varepsilon$-greedy 決策 →
部分回饋樣本記錄 → 正式模擬。標示「同 DL-THR」者為逐位元相同的環節。


## 階段 1：Spread、Z-Score 與 12 維特徵（同 DL-THR）

價格清理、Z-Score 的形成期凍結參數計算（路徑 B）、12 維特徵的定義與截尾，
全部沿用 DL-THR。特徵函式以 `_pair_features = staticmethod(_DLThreshold._pair_features)`
直接引用同一份實作。

> 這一層若各寫一份，任一邊日後被改動就會悄悄毀掉單一變因對照。


## 階段 2：$\varepsilon$-greedy 決策

$$a_t = \begin{cases}
\text{均勻抽自 } \mathcal{A} & \text{機率 } \varepsilon \\
\arg\max_a \ \text{網路預測報酬}(f_t)_a & \text{機率 } 1-\varepsilon,\ \text{且可用樣本} \ge 200 \\
(2.0,\ 0.0) & \text{機率 } 1-\varepsilon,\ \text{樣本不足}
\end{cases}$$

**探索判斷在「樣本是否足夠」之前**——這是必要的：未訓練的網路其 $\arg\max$ 沒有意義，
但若因此整段暖身期都選基準，其餘八個動作將永無樣本。

**$\varepsilon$ 排程**（三組獨立跑，各自持有網路與經驗，不互相餵食樣本）：

| 代號 | 排程 | 用意 |
| :--- | :--- | :--- |
| **E05** | $\varepsilon = 0.05$ 常數 | 低探索：貼近貪婪，但長期仍持續蒐集 |
| **E10** | $\varepsilon = 0.10$ 常數 | 中探索 |
| **E20D** | $0.20 \to 0.02$，2000 次決策線性衰減 | 前期重探索、後期重利用（bandit 的標準作法） |

::: {.callout-note}

### 為何要掃三組而非只跑一組

bandit 表現差可能有兩個彼此糾纏的原因：**資訊量只有 DL-THR 的 $1/9$**，
以及**探索期真金白銀的虧損**。$\varepsilon$ 太小則學不動、太大則被探索成本拖垮。

掃三組並以**對 bandit 最有利者**與 DL-THR 對比，可使
「反事實標籤有價值」的結論保守而站得住——否則口試上
「你怎麼知道不是 $\varepsilon$ 沒調好」將無法回答。

:::


## 階段 3：部分回饋樣本與遮罩損失

**只觀測選中動作**。以與 DL-THR 產生標籤完全相同的路徑（`_fast_threshold_pnl`）計算，
差別僅在這裡只跑一個動作、那裡跑九個：

$$r_{a_t} = \frac{\text{損益}_{a_t}}{C_{\text{每配對}}} \times 100, \qquad
\text{樣本} = (f_t,\ a_t,\ r_{a_t},\ t_e)$$

SKIP 的報酬恆為 $0$，不需模擬。

**遮罩 MSE**：每筆樣本只知道一個動作的報酬，故僅對該動作對應的輸出單元計損失：

$$\mathcal{L} = \frac{1}{|B|}\sum_{t \in B}
\Big(\underbrace{\text{網路}(f_t)_{a_t}}_{\text{只取第 } a_t \text{ 格}} - r_{a_t}\Big)^2$$

其餘八個輸出單元不接收梯度。對照 DL-THR 的全 9 維 MSE：

$$\mathcal{L}_{\text{DL-THR}} = \frac{1}{|B|}\sum_{t \in B}
\frac{1}{9}\sum_{a \in \mathcal{A}} \big(\text{網路}(f_t)_a - r_a\big)^2$$

> **資訊量差距**：同樣跑完 295 期，DL-THR 累積的「動作—報酬」觀測數是
> RL-THR 的 **9 倍**。這正是本對照要量化的東西。


## 階段 4：為何沒有折扣因子 $\gamma$

::: {.callout-important}

### 這是 contextual bandit，不是序貫 MDP——而且這件事必須明說

強化學習之所以需要 $\gamma$ 與 bootstrapped target（$r + \gamma \max_{a'} Q(s', a')$），
是因為**當前動作會改變後續狀態**，須做序貫信用分配。

本問題不具備這個性質：

1. 每組配對**每期只做一次決策**，決策後整期交由固定狀態機執行；
2. 12 維狀態**全部由形成期視窗算出**——選哪個門檻**不會改變下一期的狀態**。

沒有狀態轉移，就沒有東西可以 bootstrap。硬加 $\gamma$ 只是裝飾。

本策略的「RL 性」來自**部分回饋 + 探索／利用權衡**（Sutton & Barto 第 2 章），
而非 TD 學習。論文一律以此界定，不宣稱序貫決策。

:::

::: {.callout-tip}

### 真正的序貫 RL 已被系統性證偽

逐日定位動作空間（每天自由決定持倉）才是真正的序貫 MDP，本研究做過三代：
v1 online DQN、v2 修復版、v3 FQI（$\gamma = 0.99$，帶 bootstrapped target）。

v3 FQI 修好了 v1 的訓練與統計缺陷，但 OOS 換手 3,700+ 次（基準 617），
中位 Sharpe $-1.1 \sim -2.3$，被費用磨死。**失敗根因是動作空間設計，不是訓練方法**。

程式碼保留於 `archive/trading/drl_fqi_trading.py`，紀錄見
`archive/config_archived_strategies.py`。

:::


## 階段 5：正式模擬（同 DL-THR）

選定 $(e, x)$ 後整期執行標準 Z-Score 狀態機，費用會計、最後一日不開新倉、
輸出欄位全部與 DL-THR 及 Z-Score 基準一致。

**探索的成本計入績效**：$\varepsilon$ 抽中時實際執行的就是那個隨機動作，
其損益如實進入交易紀錄。這是誠實的 bandit——探索不是免費的模擬，而是真的下單。


# 實驗設計

| 項目 | 設定 |
| :--- | :--- |
| 配對底 | **Grid (AGG-SSD)** 一個，直接對接 `Grid (AGG-SSD-DRL)` |
| $\varepsilon$ 排程 | 三組（E05 / E10 / E20D） |
| 參數網格 | 15 格（Top N $\times$ 停損），與所有策略同 |
| 獨立重跑 | 5 輪，不固定種子 |
| `db_method` | `Grid (AGG-SSD-RLTHR-E05 / E10 / E20D)` |
| 重跑指令 | `python tools/run_drl_variance.py --module rl_threshold_trading --runs 5`（環境變數 `DRL_VARIANCE_TAG=rlthr`） |

**範圍聲明**：主張的是**學習方法**（全資訊 vs 部分回饋）而非配對底，
故一個配對底即足以回答；跨配對底的一致性未經檢定，見第五章限制。


# 結果

## 等權組合逐日對照（主口徑）

15 格等權、逐日報酬差、循環 block bootstrap（$L$=126，10,000 次），
配對底 `Grid (AGG-SSD)`，$n$ = 6,287 個交易日。

| 對照 | 年化Δ | 95% CI (pp) | $p$ (bootstrap) | NW $t$ | NW $p$ |
| :--- | ---: | :---: | ---: | ---: | ---: |
| **DL-THR − Z-Score**（參照） | **+0.787 pp** | [+0.34, +1.27] | **0.0011** | 2.995 | 0.0027 |
| RL-THR ($\varepsilon$=0.05) − Z-Score | +0.707 pp | [+0.10, +1.45] | **0.0400** | 2.395 | 0.0166 |
| **RL-THR ($\varepsilon$=0.10) − Z-Score** | **+0.802 pp** | [+0.20, +1.54] | **0.0191** | 2.703 | 0.0069 |
| RL-THR ($\varepsilon$=0.20→0.02) − Z-Score | +0.669 pp | [+0.08, +1.38] | **0.0450** | 2.332 | 0.0197 |
| RL-THR ($\varepsilon$=0.05) − DL-THR | −0.081 pp | [−0.47, +0.34] | 0.6935 | −0.431 | 0.6667 |
| **RL-THR ($\varepsilon$=0.10) − DL-THR** | **+0.015 pp** | **[−0.38, +0.45]** | **0.9409** | 0.073 | 0.9415 |
| RL-THR ($\varepsilon$=0.20→0.02) − DL-THR | −0.118 pp | [−0.49, +0.28] | 0.5357 | −0.614 | 0.5389 |

腳本：`python -m analysis.prop2_label_information` → `results/analysis/prop2_label_information.csv`

::: {.callout-important}

### 反事實標籤幾乎買不到東西

三組 $\varepsilon$ 全部給出同一個答案：RL-THR 與 DL-THR 的差距在
−0.12 ~ +0.02 pp 之間，$p$ = 0.54 ~ 0.94，無一接近顯著。

關鍵在**區間寬度而非 $p$ 值**。最有利排程下兩臂相差 **+0.015 pp**，
95% CI **[−0.38, +0.45] pp** 幾乎對稱橫跨零，
兩端都遠小於 DL-THR 對 Z-Score 的總增益 **+0.787 pp**。依本研究的判準
（區間涵蓋 0 且兩端皆小 → 方可討論「相當」；兩端皆大 → 只能說檢定力不足），
這是**真正的「兩者相當」**，不是資料不足以區分。

→ 交易層 的增益**不來自**「這個問題碰巧是全資訊的」。
把答案卷收走、改成真正的部分回饋，增益幾乎原封不動。

:::

## 五輪重跑的最佳格對照（次口徑）

`python tools/run_drl_variance.py --module rl_threshold_trading --runs 5`

| 策略 | 最佳年化中位 | 全距 | 最佳 Sharpe 中位 | 全距 | 正 Sharpe 最少 |
| :--- | ---: | :---: | ---: | :---: | :---: |
| **DL-THR**（`Grid (AGG-SSD-DRL)`） | **2.387%** | [2.341, 2.649] | **0.343** | [0.337, 0.377] | 6/15 |
| RL-THR $\varepsilon$=0.10 | 2.107% | [1.985, 2.192] | 0.307 | [0.288, 0.314] | 5/15 |
| RL-THR $\varepsilon$=0.20→0.02 | 1.924% | [1.777, 2.058] | 0.286 | [0.262, 0.299] | 6/15 |
| RL-THR $\varepsilon$=0.05 | 1.897% | [1.811, 2.045] | 0.276 | [0.266, 0.294] | 6/15 |

::: {.callout-warning}

### 兩個口徑給出相反的答案——而這正是報告口徑重要性的實例

**最佳格**上 DL-THR 五輪全勝且全距完全不重疊（DL 最低 2.341% > RL 最高 2.192%；
Sharpe 亦然，0.337 > 0.314）。**等權組合**上兩者無法區分。

不矛盾：best-of-15 會放大微小且系統性的優勢——資訊量九倍的那一臂更**可靠地**
產出好的極大值，即使其平均水準相同。等權組合把這個選擇效應平均掉。

本研究一律以等權組合為報告口徑（見 §3.5），故結論取「兩者相當」。
但這組數字同時說明：**若改報最佳格，同一份資料會支持相反的結論。**

:::

## 判讀與限制

**RL-THR 對 Z-Score 三組 $\varepsilon$ 全部顯著**（0.019 / 0.040 / 0.045）。
2026-08-10 重跑前有兩組僅為邊緣顯著（0.058 / 0.068），故當時另列
「bandit 自身顯著性較脆弱」為限制；重跑後該限制已不成立。

**bootstrap 與 NW 對 $\varepsilon$=0.05／衰減排程結論不一致**（NW 顯著、bootstrap 不顯著）。
主檢定以 bootstrap 為準；差異源於 bootstrap 的雙尾 $p$ 對分布偏態較敏感，
在 $p\approx0.05$ 邊界附近可與 CI 產生輕微不一致（見 `analysis/block_bootstrap` docstring）。

**兩個效應無法分離。** 差距同時含「資訊量僅 1/9」與「探索期真金白銀的虧損」。
不探索就沒有樣本——這是部分回饋的定義，不是實作可以繞開的。
掃三組 $\varepsilon$ 界定了取捨曲線，但不能拆成兩個獨立的數字。

**單一配對底、單一訓練輪的逐日對照。** 等權逐日檢定取自 `result.db` 的最後一輪
（`trade_logs` 只保留最新一輪）；輪間變異另由上表的五輪統計呈現。
跨配對底的一致性未檢定。


# 參數總表

| 參數 | 值 | 對應環節 | 說明 |
| :--- | :---: | :--- | :--- |
| 動作選單 | 跳過 + 進場{1.5,2,2.5,3}×出場{0,0.5} | 決策 | 共 9 個，同 DL-THR |
| 基準（保底）門檻 | 偏離 2.0 進場、回歸 0 出場 | 決策 | 僅作用於**貪婪臂**；探索臂不受限 |
| 探索率 $\varepsilon$ | 0.05 / 0.10 / 0.20→0.02 | 決策 | 三組獨立跑，各持有獨立網路與經驗 |
| 衰減步數 | 2000 次決策 | 決策 | 僅 E20D 使用 |
| 神經網路寬度 | 兩層各 64 | 決策 | 同 DL-THR |
| 學習率 | 0.001 | 學習 | 同 DL-THR |
| 每次訓練輪數 | 40 | 學習 | 同 DL-THR |
| 啟用學習的最低樣本 | 200 | 學習 | 同 DL-THR；但**不阻擋探索** |
| 損失 | 遮罩 MSE（1/9 維） | 學習 | **與 DL-THR 的關鍵差異** |
| 特徵維度 | 12 | 特徵 | 共用 DL-THR 的 `_pair_features` |
| 隨機種子 | 不固定 | 學習/決策 | 兩層隨機性，以五輪重跑評估 |
| 每配對資金 / 往返成本 | 10,000 / 0.58% | 執行 | 與所有策略相同 |
